# Similarity and Topic Diversity - Regression

In [1]:
import statsmodels.api as sm
import numpy as np
import pandas as pd

INPUT_PATH = 'home/MoviesLSE/topic_diversity_imdb_eda.csv'
VAR_LIST = ['imdb_id', 'title', 'n_sentences', 'mean_cosine_similarity', 'TDI',
       'dominant_topic', 'startYear', 'runtimeMinutes', 'genres', 'averageRating', 'numVotes', 'log_votes',
       'era', 'age_cert', 'box_office']


df = pd.read_csv(INPUT_PATH, usecols=VAR_LIST)

In [4]:
# ── Reference categories ──────────────────────────────────────────────────────
REF = {
    "genres":    "Action",   # drop_first drops alphabetically first
    "age_cert":  "G",
    "titleType": "movie",
}

# 0. Normalize

for col in ['mean_cosine_similarity', 'n_sentences', 'log_votes', 'box_office']:
    df[col] = (df[col]-df[col].mean())/df[col].std()

# 1. Dummies
genres_dummies = df["genres"].str.get_dummies(sep=",")
age_dummies    = pd.get_dummies(df["age_cert"],  prefix="age_cert",  drop_first=True)

merged_df = pd.concat([df, genres_dummies, age_dummies], axis=1)

# 2. Keep dummies with ≥50 obs
all_dummy_cols = list(genres_dummies.columns) + list(age_dummies.columns)
dummy_cols     = [c for c in all_dummy_cols if merged_df[c].sum() >= 50]

base_features  = ["num_sentences", "startYear", "runtimeMinutes", "log_votes", "TDI",  "mean_cosine_similarity", 'box_office']
feature_cols   = base_features + dummy_cols



# ── Build dummies & model data (same as before) ───────────────────────────────

all_dummy_cols = list(genres_dummies.columns) + list(age_dummies.columns) 
dummy_cols     = [c for c in all_dummy_cols if merged_df[c].sum() >= 50]

genre_cols = set(genres_dummies.columns) & set(dummy_cols)
age_cols   = {c for c in dummy_cols if c.startswith("age_cert_")}


# ── Stars helper ──────────────────────────────────────────────────────────────
def stars(p):
    if p < 0.001: return "***"
    elif p < 0.01:  return "**"
    elif p < 0.05:  return "*"
    elif p < 0.1:   return "."
    else:           return ""

# ── Pretty-print with reference rows as actual table rows ────────────────────
def print_model(model, title, ref_dict, genre_cols, age_cols):
    print(f"\n{'='*90}")
    print(f"  {title}")
    print(f"  R² = {model.rsquared:.4f}  |  Adj. R² = {model.rsquared_adj:.4f}"
          f"  |  F-stat p = {model.f_pvalue:.2e}")
    print(f"{'='*90}")
    print(f"  {'Variable':<38} {'coef':>9} {'std err':>9} {'t':>8} {'P>|t|':>8} {'sig':>5}")
    print(f"  {'-'*83}")

    rows = list(zip(model.params.index, model.params, model.bse,
                    model.tvalues, model.pvalues))

    first_genre = True
    first_age   = True
    first_type  = True

    for name, coef, se, t, p in rows:

        # Before first genre dummy → insert reference row
        if name in genre_cols and first_genre:
            print(f"  {'-'*83}")
            print(f"  {'[Genre dummies]':<38} {'ref: ' + ref_dict['genres']:>9}")
            print(f"  {ref_dict['genres']:<38} {'1.0000':>9} {'(base)':>9} {'—':>8} {'—':>8} {'—':>5}")
            first_genre = False

        # Before first age_cert dummy → insert reference row
        if name.startswith("age_cert_") and first_age:
            print(f"  {'-'*83}")
            print(f"  {'[Age cert dummies]':<38} {'ref: ' + ref_dict['age_cert']:>9}")
            print(f"  {ref_dict['age_cert']:<38} {'1.0000':>9} {'(base)':>9} {'—':>8} {'—':>8} {'—':>5}")
            first_age = False
        print(f"  {name:<38} {coef:>9.4f} {se:>9.4f} {t:>8.3f} {p:>8.4f} {stars(p):>5}")

    print(f"  {'-'*83}")
    print("  Significance: *** p<0.001  ** p<0.01  * p<0.05  . p<0.1")
    print(f"{'='*90}\n")
    


# ── MODEL 1 — DV: averageRating ───────────────────────────────────────────────
base1  = ["startYear", "runtimeMinutes", "n_sentences", "log_votes", "box_office", "TDI", "mean_cosine_similarity"]
feat1  = base1 + dummy_cols
X1 = merged_df[feat1].astype(float).dropna()
y1 = pd.to_numeric(merged_df.loc[X1.index, "averageRating"], errors="coerce")
mask1 = y1.notna(); X1, y1 = X1[mask1], y1[mask1].astype(float)
m1 = sm.OLS(y1, sm.add_constant(X1)).fit()
print_model(m1, "MODEL 1 — DV: averageRating", REF, genre_cols, age_cols)

# ── MODEL 2 — DV: mean_cosine_similarity ─────────────────────────────────────
base2  = ["startYear", "runtimeMinutes", "n_sentences", "log_votes", "box_office"]
feat2  = base2 + dummy_cols
X2 = merged_df[feat2].astype(float).dropna()
y2 = pd.to_numeric(merged_df.loc[X2.index, "mean_cosine_similarity"], errors="coerce")
mask2 = y2.notna(); X2, y2 = X2[mask2], y2[mask2].astype(float)
m2 = sm.OLS(y2, sm.add_constant(X2)).fit()
print_model(m2, "MODEL 2 — DV: Mean Cosine Similarity", REF, genre_cols, age_cols)

# ── MODEL 3 — DV: avg_cosine_similarity ──────────────────────────────────────
base3  = [ "startYear", "runtimeMinutes", "n_sentences", "log_votes", "box_office"]
feat3  = base3 + dummy_cols
X3 = merged_df[feat3].astype(float).dropna()
y3 = pd.to_numeric(merged_df.loc[X3.index, "TDI"], errors="coerce")
mask3 = y3.notna(); X3, y3 = X3[mask3], y3[mask3].astype(float)
m3 = sm.OLS(y3, sm.add_constant(X3)).fit()
print_model(m3, "MODEL 3 — DV: Topic Diversity Index", REF, genre_cols, age_cols)


  MODEL 1 — DV: averageRating
  R² = 0.6341  |  Adj. R² = 0.6217  |  F-stat p = 6.23e-125
  Variable                                    coef   std err        t    P>|t|   sig
  -----------------------------------------------------------------------------------
  const                                   -30.3889    8.4750   -3.586   0.0004   ***
  startYear                                 0.0193    0.0042    4.566   0.0000   ***
  runtimeMinutes                            0.0078    0.0014    5.448   0.0000   ***
  n_sentences                              -0.0022    0.0266   -0.083   0.9342      
  log_votes                                 0.8141    0.0381   21.385   0.0000   ***
  box_office                               -0.0710    0.0284   -2.499   0.0127     *
  TDI                                      -2.0200    1.6277   -1.241   0.2151      
  mean_cosine_similarity                   -0.0301    0.0313   -0.963   0.3360      
  --------------------------------------------------------